# Firefox Bugzilla Comment Analysis

Exploratory data analysis of Firefox bug report comments from Mozilla's Bugzilla tracker.

This notebook covers:
1. Data loading and cleaning
2. Comment volume and activity patterns
3. Sentiment analysis of bug comments
4. Topic modeling
5. Contributor analysis
6. Response time statistics

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_collector import fetch_bugs, fetch_bug_comments
from src.text_analysis import analyze_sentiment, extract_topics, extract_keywords, get_word_frequencies, comment_length_stats
from src.visualizations import (
    plot_comment_volume, plot_sentiment_distribution, plot_sentiment_trend,
    plot_contributor_activity, plot_topic_distribution, plot_comment_length_dist,
    plot_wordcloud, plot_activity_heatmap, create_summary_dashboard
)
from src.stats import (
    bug_summary_stats, comment_summary_stats, response_time_analysis,
    contributor_analysis, weekly_activity_summary
)

%matplotlib inline
sns.set_theme(style='whitegrid')
print('Imports loaded.')

## 1. Data Collection

Fetch Firefox bugs and comments from the Bugzilla REST API for 2021.
If data files already exist locally, load from CSV instead.

In [ ]:
import os

YEAR = 2021
DATA_DIR = '../data'
BUGS_FILE = os.path.join(DATA_DIR, f'firefox_bugs_{YEAR}.csv')
COMMENTS_FILE = os.path.join(DATA_DIR, f'firefox_comments_{YEAR}.csv')

if os.path.exists(BUGS_FILE) and os.path.exists(COMMENTS_FILE):
    print('Loading from local CSV files...')
    bugs_df = pd.read_csv(BUGS_FILE, parse_dates=['creation_time', 'last_change_time'])
    comments_df = pd.read_csv(COMMENTS_FILE, parse_dates=['created'])
else:
    print('Fetching from Bugzilla API...')
    bugs_df = fetch_bugs(product='Firefox', start_date=f'{YEAR}-01-01', end_date=f'{YEAR}-12-31')
    bug_ids = bugs_df['id'].tolist()
    comments_df = fetch_bug_comments(bug_ids)

    os.makedirs(DATA_DIR, exist_ok=True)
    bugs_df.to_csv(BUGS_FILE, index=False)
    comments_df.to_csv(COMMENTS_FILE, index=False)

print(f'Bugs: {len(bugs_df)}, Comments: {len(comments_df)}')
bugs_df.head()

In [ ]:
comments_df.head()

## 2. Bug Overview Statistics

In [ ]:
bug_stats = bug_summary_stats(bugs_df)

print(f"Total bugs: {bug_stats['total_bugs']}")
print(f"Unique reporters: {bug_stats.get('unique_reporters')}")
print(f"Resolution rate: {bug_stats.get('resolution_rate', 0):.1%}")
print(f"Date range: {bug_stats.get('date_range')}")
print(f"\nStatus breakdown:")
for status, count in bug_stats.get('status_counts', {}).items():
    print(f"  {status}: {count}")

In [ ]:
# top components
if 'component' in bugs_df.columns:
    fig, ax = plt.subplots(figsize=(10, 6))
    bugs_df['component'].value_counts().head(15).plot.barh(ax=ax, color='steelblue')
    ax.set_title('Top 15 Components by Bug Count')
    ax.set_xlabel('Number of Bugs')
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 3. Comment Volume and Activity Patterns

In [ ]:
comment_stats = comment_summary_stats(comments_df)
print(f"Total comments: {comment_stats['total_comments']}")
print(f"Unique authors: {comment_stats.get('unique_authors')}")
print(f"Avg words per comment: {comment_stats.get('avg_word_count', 0):.1f}")
print(f"Comments per bug (mean): {comment_stats.get('comments_per_bug_mean', 0):.1f}")
print(f"Comments per bug (median): {comment_stats.get('comments_per_bug_median', 0):.1f}")

In [ ]:
plot_comment_volume(comments_df, freq='W', title='Weekly Comment Volume (2021)')
plt.show()

In [ ]:
plot_comment_length_dist(comments_df)
plt.show()

In [ ]:
plot_activity_heatmap(comments_df)
plt.show()

## 4. Sentiment Analysis

In [ ]:
sentiment_df = analyze_sentiment(comments_df)

print(f"Mean polarity: {sentiment_df['polarity'].mean():.3f}")
print(f"Mean subjectivity: {sentiment_df['subjectivity'].mean():.3f}")
print(f"\nSentiment label counts:")
print(sentiment_df['sentiment_label'].value_counts())

In [ ]:
plot_sentiment_distribution(sentiment_df)
plt.show()

In [ ]:
plot_sentiment_trend(sentiment_df)
plt.show()

In [ ]:
# sentiment by component
if 'bug_id' in sentiment_df.columns and 'component' in bugs_df.columns:
    merged = sentiment_df.merge(bugs_df[['id', 'component']], left_on='bug_id', right_on='id', how='left')
    comp_sentiment = merged.groupby('component')['polarity'].agg(['mean', 'count'])
    comp_sentiment = comp_sentiment[comp_sentiment['count'] >= 20].sort_values('mean')
    
    fig, ax = plt.subplots(figsize=(10, 6))
    comp_sentiment['mean'].plot.barh(ax=ax, color=['#e74c3c' if x < 0 else '#2ecc71' for x in comp_sentiment['mean']])
    ax.set_title('Mean Sentiment by Component')
    ax.set_xlabel('Mean Polarity')
    ax.axvline(0, color='gray', linestyle='--')
    plt.tight_layout()
    plt.show()

## 5. Topic Modeling

In [ ]:
topic_results = extract_topics(comments_df, n_topics=8, method='lda')

print('Discovered Topics:\n')
for topic_name, words in topic_results['topic_words'].items():
    print(f"{topic_name}: {', '.join(words[:10])}")

In [ ]:
plot_topic_distribution(topic_results['topic_words'], topic_results['doc_topics'])
plt.show()

In [ ]:
# keyword extraction
keywords = extract_keywords(comments_df, top_n=40)
print('Top keywords by TF-IDF score:')
for word, score in keywords[:20]:
    print(f'  {word:30s} {score:.4f}')

In [ ]:
word_freq = get_word_frequencies(comments_df, top_n=150)
plot_wordcloud(word_freq, title='Most Common Words in Bug Comments')
plt.show()

## 6. Contributor Analysis

In [ ]:
contrib_stats = contributor_analysis(comments_df)

print(f"Total contributors: {contrib_stats['total_contributors']}")
print(f"Gini coefficient: {contrib_stats['gini_coefficient']:.3f}")
print(f"Top 1% share: {contrib_stats['top_1pct_share']:.1%}")
print(f"Top 5% share: {contrib_stats['top_5pct_share']:.1%}")
print(f"Top 10% share: {contrib_stats['top_10pct_share']:.1%}")
print(f"Single-comment authors: {contrib_stats['single_comment_authors']}")

In [ ]:
plot_contributor_activity(comments_df, top_n=20)
plt.show()

In [ ]:
weekly = weekly_activity_summary(comments_df)

fig, axes = plt.subplots(3, 1, figsize=(14, 10), sharex=True)

weekly['comment_count'].plot(ax=axes[0], color='steelblue')
axes[0].set_ylabel('Comments')
axes[0].set_title('Weekly Activity Trends')

weekly['unique_authors'].plot(ax=axes[1], color='coral')
axes[1].set_ylabel('Active Contributors')

weekly['unique_bugs'].plot(ax=axes[2], color='seagreen')
axes[2].set_ylabel('Active Bugs')
axes[2].set_xlabel('Week')

plt.tight_layout()
plt.show()

## 7. Response Time Analysis

In [ ]:
rt_stats = response_time_analysis(comments_df)

fr = rt_stats['first_response']
print('First Response Time:')
print(f"  Mean: {fr['mean_hours']:.1f} hours")
print(f"  Median: {fr['median_hours']:.1f} hours")
print(f"  90th percentile: {fr['p90_hours']:.1f} hours")
print(f"  Responded within 1h: {fr['responded_within_1h']} / {fr['bugs_analyzed']}")
print(f"  Responded within 24h: {fr['responded_within_24h']} / {fr['bugs_analyzed']}")

ic = rt_stats['inter_comment']
print(f"\nInter-comment interval:")
print(f"  Mean: {ic['mean_hours']:.1f} hours")
print(f"  Median: {ic['median_hours']:.1f} hours")

## 8. Summary Dashboard

In [ ]:
create_summary_dashboard(
    comments_df,
    sentiment_df=sentiment_df,
    topic_results=topic_results,
    word_freq=word_freq,
)
plt.show()

## Key Findings

- Comment volume shows clear weekday/weekend patterns with most activity during business hours (UTC)
- Overall sentiment skews slightly positive, though technical discussions tend toward neutral
- Topic modeling reveals distinct clusters around crashes/stability, UI/UX, web compatibility, and developer tooling
- Contribution follows a heavy-tailed distribution -- a small core group produces the majority of comments
- Median first-response time indicates active triage, though some components lag behind others